# Week 20 - MLOps CI/CD and Monitoring (Data Engineer Variant, SageMaker)

Last week you used SageMaker MLflow and the Model Registry to ship a
fraud-detection model behind the endpoint `week19-fraud-endpoint`. This week
we shift our attention to what data engineers actually own in production:
the PIPELINE that feeds that model.

If the input distribution silently changes, the model still answers, just
wrongly. The data engineer is the first line of defense.

## What you will build today

1. An S3-based freshness monitor that watches the fraud CSV landing in
   `s3://bread-academy-week19-shared/` and computes how the new batch
   differs from a rolling baseline using pandas.
2. A pandas data quality gate with four production-grade assertions, run
   before any downstream consumer touches the data.
3. An S3 Parquet audit log at
   `s3://bread-academy-week19-shared/week20/pipeline_audit_log/` so every
   pipeline run leaves a queryable trace you can join with MLflow runs
   later.
4. An automated retraining trigger that calls the SageMaker Pipelines
   API to start the ML engineer's retraining job when your data quality
   signals fire.

## Learning objectives

- Detect freshness and drift in S3-resident tabular data using pandas.
- Express data quality checks as pandas assertions you can run in CI or a
  SageMaker Processing Job.
- Write structured audit events to a partitioned S3 Parquet dataset.
- Trigger SageMaker Pipelines programmatically with the SDK.

## Environment Setup

**Platform**: AWS SageMaker Studio (di-mfa account, us-east-1).

**Kernel**: Data Science 3.0 (Python 3.10) on a `ml.t3.medium` instance is
sufficient - we do all work in pandas, no GPU and no Spark.

**Required libraries**: All pinned via `%pip install` in the next cell.

**Auth**: The notebook uses the SageMaker execution role attached to your
Studio user. No getpass, no static AWS keys. The role already has:
- Read access to `s3://bread-academy-week19-shared/`
- Write access to `s3://bread-academy-week19-shared/week20/`
- `sagemaker:StartPipelineExecution` on the stub pipeline
  `week20-retrain-stub`
- Bedrock invoke access to `us.anthropic.claude-sonnet-4-5-20250929-v1:0`
  (kept for parity with prior weeks; not used in this notebook)

In [ ]:
%pip install --quiet \
    "sagemaker==2.257.3" \
    "mlflow==3.10.0" \
    "boto3>=1.35,<2" \
    "pandas>=2.0,<3" \
    "scipy>=1.11,<2" \
    "pyarrow>=14,<18" \
    "s3fs>=2024.3"

from importlib.metadata import version
import os, json, time, uuid, io
from datetime import datetime, timedelta, timezone
import boto3
import pandas as pd
import numpy as np
import sagemaker
from sagemaker import get_execution_role

print("sagemaker:", version("sagemaker"))
print("boto3:", version("boto3"))
print("pandas:", version("pandas"))
print("scipy:", version("scipy"))
print("pyarrow:", version("pyarrow"))

In [ ]:
# SageMaker auth - this is the standard di-mfa pattern from Weeks 15-19.
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
print("Region:", AWS_REGION)
print("Role:", role)

s3 = boto3.client("s3", region_name=AWS_REGION)

# Shared bucket and paths reused across the week
SHARED_BUCKET = "bread-academy-week19-shared"
SOURCE_KEY = "raw/fraud_transactions.csv"
AUDIT_PREFIX = "week20/pipeline_audit_log"
SOURCE_S3 = f"s3://{SHARED_BUCKET}/{SOURCE_KEY}"

# Drift simulation: the CSV has partition_date to separate baseline (days 0-59,
# merchant_country US rate ~0.80) from the drift window (days 60-89, US rate
# ~0.55, fraud rate climbs from 3% to 4.5%). We slice by partition_date below.
AUDIT_S3 = f"s3://{SHARED_BUCKET}/{AUDIT_PREFIX}"

# Reused from Week 19
ENDPOINT_NAME = "week19-fraud-endpoint"   # consumer downstream, not modified here
RETRAIN_PIPELINE_NAME = "week20-retrain-stub"

# Bedrock model id (parity with Weeks 15-19; not invoked in this notebook).
LLM_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("Source CSV:", SOURCE_S3)
print("Audit prefix:", AUDIT_S3)

In [ ]:
# Probe 1: source CSV is reachable
head = s3.head_object(Bucket=SHARED_BUCKET, Key=SOURCE_KEY)
print(f"Source CSV OK: {head['ContentLength']:,} bytes, "
      f"last modified {head['LastModified']}.")

# Probe 2: audit prefix is writable
probe_key = f"{AUDIT_PREFIX}/_probe_w20.txt"
try:
    s3.put_object(Bucket=SHARED_BUCKET, Key=probe_key, Body=b"probe")
    s3.delete_object(Bucket=SHARED_BUCKET, Key=probe_key)
    print(f"Audit prefix WRITE OK: s3://{SHARED_BUCKET}/{AUDIT_PREFIX}")
except Exception:
    print("Ask your instructor to grant s3:PutObject on "
          f"s3://{SHARED_BUCKET}/{AUDIT_PREFIX}/")
    raise

# Probe 3: retraining pipeline is reachable (does NOT start it)
sm = boto3.client("sagemaker", region_name=AWS_REGION)
try:
    sm.describe_pipeline(PipelineName=RETRAIN_PIPELINE_NAME)
    print(f"Pipeline OK: {RETRAIN_PIPELINE_NAME}")
except Exception:
    print(f"Ask your instructor to create pipeline {RETRAIN_PIPELINE_NAME}.")
    raise

## What Are We Building Today?

Imagine you are on call for the Bread Financial fraud platform. At 3am the
on-call ML engineer pages you: "the model is flagging twice as many txns as
yesterday, did something change upstream?" Without pipeline observability
your only answer is to start downloading CSVs and grepping them by hand.
With it, you already know: the source CSV in S3 grew by 1.4M new rows in
the last 6 hours (normal: 800k), the merchant_category mix shifted hard
toward category 42 (a new partner went live), and the PSI score on
`amount` is 0.31 (way above the 0.2 alert threshold).

You did not write the model. You did not retrain it. But you SAVED the
model owner from a bad night because the pipeline told its own story.

That is what the next four topics build, end to end.

## Topic 1 - S3 freshness monitor on the fraud CSV

### Why a freshness monitor at all?

In a Databricks shop you would use Delta Change Data Feed to read exactly
which rows arrived since version N. We do not have Delta here - the
upstream system writes a CSV (or set of CSVs) into S3. We replace CDF with
two simple S3-native signals:

1. The S3 object's `LastModified` timestamp tells us when the upstream
   producer last refreshed the data.
2. A `recent_window` slice of the CSV (rows whose `partition_date` integer
   day offset >= 60) plays the role of the "inserts since version N" slice.
   Days 0-59 are the baseline (US merchant_country rate ~0.80, fraud ~3%).
   Days 60-89 are the drift window (US rate ~0.55, fraud climbs to ~4.5%).
   This is the population we compare against the baseline for drift detection.

This is the standard pattern for S3-as-source pipelines: use file
metadata for freshness, and an in-data partition column for slicing.

### Loading the CSV

```python
df = pd.read_csv(SOURCE_S3)
print(df.shape)
print("partition_date range:", df["partition_date"].min(), "to", df["partition_date"].max())
```

The whole CSV fits comfortably in memory at this scale (about 1 GB
uncompressed). For a real production pipeline at TB scale you would
push this exact logic into a SageMaker Processing Job and process in
chunks.

In [ ]:
# Pull the S3 object's LastModified for the freshness signal
head = s3.head_object(Bucket=SHARED_BUCKET, Key=SOURCE_KEY)
last_modified = head["LastModified"]
print("Source last modified:", last_modified)

# Load the full table (about 1 GB, fits in t3.medium memory).
# partition_date is an integer day offset (0-89) in the simulation CSV.
df = pd.read_csv(SOURCE_S3)

# Defensive dtype coercion: pd.read_csv may infer partition_date as object/string
# if the CSV header or any row is ambiguous. Force numeric so downstream
# comparisons like `df["partition_date"] >= 60` cannot raise TypeError.
df["partition_date"] = pd.to_numeric(df["partition_date"], errors="raise").astype("int64")
print("Rows:", len(df), "  Cols:", list(df.columns))
print("partition_date dtype:", df["partition_date"].dtype,
      " range:", int(df["partition_date"].min()), "to", int(df["partition_date"].max()))

# Slice the drift window (partition_date >= 60) as the "recent window".
# Days 0-59 = baseline (US merchant_country rate ~0.80, fraud rate ~3%).
# Days 60-89 = drift window (US rate drops to ~0.55, fraud climbs to ~4.5%).
DRIFT_WINDOW_START = 60
recent_mask = df["partition_date"] >= DRIFT_WINDOW_START
print(f"Drift window (partition_date >= {DRIFT_WINDOW_START}): "
      f"{recent_mask.sum():,} rows of {len(df):,} total")

df.loc[recent_mask, ["partition_date", "amount", "is_fraud"]].head(10)

## Lab 1 - Build a freshness monitor (10 minutes)

Build a function `freshness_snapshot(df, recent_mask, last_modified)` that
returns a single Python dict with these keys:

- `rows_arrived` - count of rows in the drift window (recent_mask == True).
- `last_modified` - the S3 object's `LastModified` timestamp.
- `fraud_rate_recent` - fraction of drift-window rows where `is_fraud == 1`.
- `fraud_rate_baseline` - overall fraud rate across the full table.
- `fraud_rate_delta` - `fraud_rate_recent - fraud_rate_baseline`.

Then log all five values to MLflow under experiment `MLFLOW_EXP` as a
single run named `freshness-<timestamp>`. You can use the lightweight
local MLflow tracking that ships with SageMaker Studio
(`mlflow.set_tracking_uri("file:./mlruns")`).

You can reuse `df`, `recent_mask`, and `last_modified` from the demo cell.

In [ ]:
import mlflow
mlflow.set_tracking_uri("file:./mlruns")
MLFLOW_EXP = "week20_pipeline_health"

def freshness_snapshot(df, recent_mask, last_modified) -> dict:
    # YOUR CODE
    snapshot = None
    return snapshot

mlflow.set_experiment(MLFLOW_EXP)
with mlflow.start_run(run_name=f"freshness-{int(time.time())}"):
    snap = None  # YOUR CODE
    # YOUR CODE

print(snap)

In [ ]:
# SAFETY-NET for Lab 1. Run this only if you did NOT finish Lab 1.
if 'snap' not in dir() or snap is None:
    print("Using Lab 1 safety-net.")
    recent = df.loc[recent_mask]
    fraud_recent = float(recent["is_fraud"].mean()) if len(recent) else 0.0
    fraud_baseline = float(df["is_fraud"].mean())
    snap = {
        "rows_arrived": int(len(recent)),
        "last_modified": last_modified,
        "fraud_rate_recent": fraud_recent,
        "fraud_rate_baseline": fraud_baseline,
        "fraud_rate_delta": fraud_recent - fraud_baseline,
    }
    print(snap)

## Think About It

You used the S3 object's `LastModified` plus an in-data window on
`partition_date`. A teammate asks: "Why not just use `LastModified`
alone? Why bother with `partition_date`?" What do you tell them?

(Hint: `LastModified` tells you when the file was WRITTEN, not when the
events INSIDE it happened. If the producer batches every 6 hours and you
poll every hour, `LastModified` will be hours stale relative to the
freshest event in the file. You need BOTH: `LastModified` for "is the
producer alive", and `partition_date` for "which slice of data is the
drift window vs. the baseline".)

## Topic 2 - A data quality gate before the model sees the data

The freshness monitor tells you WHAT changed. A data quality gate decides
whether the change is acceptable. If it is not, you HALT the pipeline so
the downstream consumer (the fraud model) never sees the bad batch.

We will use four checks that cover 80 percent of real production failures:

1. No nulls in `is_fraud` (the label column must always be present).
2. `amount > 0` on every row (no refunds, no test rows).
3. `partition_date` (integer day offset) within [0, 90] (no out-of-range rows).
4. Fraud rate between 1 percent and 20 percent (class balance sanity).

We implement them as plain pandas assertions because (a) they are easy
to read, (b) they run on the same engine as the data, and (c) they do
not add a fragile dependency on Great Expectations for a beginner
audience. In production you would lift these same functions into a
SageMaker Processing Job and run them on every new CSV before the
downstream consumer is allowed to load it.

In [ ]:
def assert_no_null_labels(df, col_name="is_fraud"):
    n_null = int(df[col_name].isna().sum())
    if n_null > 0:
        raise AssertionError(f"DQ FAIL: {n_null} null values in '{col_name}'")
    return {"check": "no_null_labels", "passed": True, "n_null": 0}

# Demo on the current CSV
result = assert_no_null_labels(df)
print(result)

## Lab 2 - Implement the remaining 3 checks (10 minutes)

Following the pattern in the demo, implement three functions. Each must
return a dict with at least `check`, `passed`, and one diagnostic field
(e.g., `n_bad`, `rate`, `min_date`). Each must raise `AssertionError`
with a clear message on failure:

1. `assert_positive_amount(df)` - all rows must have `amount > 0`.
2. `assert_recent_dates(df, days=90)` - all `partition_date` values
   (integer day offsets) must be <= `days`. The simulation uses days 0-89,
   so this catches any stray rows outside the expected range.
3. `assert_fraud_rate_in_band(df, lo=0.01, hi=0.20)` - overall fraud
   rate in `[lo, hi]`.

Then wrap all four in a `run_quality_gate(df)` function that runs them
sequentially, collects results in a list, and returns
`(all_passed, results)`. The gate must NOT short-circuit on the first
failure - we want to see every failing check in one shot.

In [ ]:
def assert_positive_amount(df):
    # YOUR CODE
    return None

def assert_recent_dates(df, days=90):
    # YOUR CODE
    return None

def assert_fraud_rate_in_band(df, lo=0.01, hi=0.20):
    # YOUR CODE
    return None

def run_quality_gate(df):
    # YOUR CODE
    return None, []

all_passed, gate_results = run_quality_gate(df)
print("ALL PASSED:", all_passed)
for r in gate_results:
    print(r)

In [ ]:
# SAFETY-NET for Lab 2.
if 'all_passed' not in dir() or all_passed is None:
    print("Using Lab 2 safety-net.")

    def _check(name, value, passed):
        return {"check": name, "passed": bool(passed), "value": value}

    def assert_positive_amount(df):
        n_bad = int((df["amount"] <= 0).sum())
        return _check("positive_amount", n_bad, n_bad == 0)

    def assert_recent_dates(df, days=90):
        # partition_date is an integer day offset (0-89 in the simulation).
        # The check ensures all rows fall within the expected day range [0, days].
        n_bad = int((df["partition_date"] > days).sum()) if "partition_date" in df.columns else 0
        return _check("recent_dates", n_bad, n_bad == 0)

    def assert_fraud_rate_in_band(df, lo=0.01, hi=0.20):
        rate = float(df["is_fraud"].mean())
        return _check("fraud_rate_band", round(rate, 4), lo <= rate <= hi)

    def run_quality_gate(df):
        results = [
            assert_no_null_labels(df),
            assert_positive_amount(df),
            assert_recent_dates(df),
            assert_fraud_rate_in_band(df),
        ]
        return all(r["passed"] for r in results), results

    all_passed, gate_results = run_quality_gate(df)
    print("ALL PASSED:", all_passed)
    for r in gate_results:
        print(r)

## Topic 3 - Structured audit log on S3 Parquet

Now we have two signals (freshness snapshot, quality gate) and no place
to put them. MLflow is fine for ad-hoc metric logging but it is awkward
to JOIN against other datasets for analytics. So we add a second store:
a partitioned Parquet dataset under
`s3://bread-academy-week19-shared/week20/pipeline_audit_log/` that
mirrors every run.

### Schema we will write

| Column | Type | Meaning |
|--------|------|---------|
| run_id | string | UUID, one per pipeline invocation |
| run_timestamp | timestamp | When the run completed |
| rows_processed | int64 | Total rows seen this run |
| fraud_rate | float64 | Observed fraud rate this run |
| psi_score | float64 | PSI of `amount` against the baseline |
| drift_detected | bool | True if any DQ check failed or PSI > 0.2 |
| action_taken | string | One of: `none`, `alerted`, `retrain_triggered` |

We partition by `run_date=YYYY-MM-DD` so Athena (or pandas with
`s3fs`) can prune old partitions quickly. The S3 bucket has object
versioning enabled (Week 19 setup), so accidental overwrites are
recoverable. The first write creates the prefix; subsequent writes
land new files alongside existing ones.

In [ ]:
def audit_key(run_id, run_ts):
    run_date = run_ts.strftime("%Y-%m-%d")
    return f"{AUDIT_PREFIX}/run_date={run_date}/{run_id}.parquet"

def write_audit_record(record: dict):
    # Force schema by going through a single-row DataFrame
    audit_df = pd.DataFrame([record]).astype({
        "run_id": "string",
        "rows_processed": "int64",
        "fraud_rate": "float64",
        "psi_score": "float64",
        "drift_detected": "bool",
        "action_taken": "string",
    })
    key = audit_key(record["run_id"], record["run_timestamp"])
    buf = io.BytesIO()
    audit_df.to_parquet(buf, engine="pyarrow", index=False)
    buf.seek(0)
    s3.put_object(Bucket=SHARED_BUCKET, Key=key, Body=buf.getvalue())
    return f"s3://{SHARED_BUCKET}/{key}"

demo_record = {
    "run_id": str(uuid.uuid4()),
    "run_timestamp": pd.Timestamp(datetime.now(timezone.utc)),
    "rows_processed": int(snap["rows_arrived"]),
    "fraud_rate": float(snap["fraud_rate_recent"]),
    "psi_score": 0.05,        # placeholder PSI, we compute the real one in Lab 3
    "drift_detected": False,
    "action_taken": "none",
}
written = write_audit_record(demo_record)
print("Wrote:", written)

# Read back the most recent partition to verify
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
preview = pd.read_parquet(
    f"s3://{SHARED_BUCKET}/{AUDIT_PREFIX}/run_date={today}/"
)
preview.sort_values("run_timestamp", ascending=False).head(5)

## Lab 3 - Compute PSI on `amount` and write a real audit row (10 minutes)

PSI (Population Stability Index) on a continuous column is computed by:

1. Bucketing the baseline distribution into N (use 10) quantile bins.
2. Computing the proportion of rows in each bin for baseline and recent.
3. PSI = sum_i (recent_i - baseline_i) * ln(recent_i / baseline_i),
   with epsilon = 1e-10 to avoid divide-by-zero on empty bins.

Implement:

- `psi_amount(baseline, recent, n_bins=10) -> float` using
  `numpy.quantile` to get the bin edges from `baseline["amount"]` and
  `numpy.histogram` (or `pandas.cut`) to bucket both populations.
- `write_audit_row(snap, psi, drift_detected, action_taken)` that
  writes a single Parquet file via `write_audit_record`.

Then call them. Use the full DataFrame as the baseline and the recent
window slice as the recent population.

In [ ]:
def psi_amount(baseline, recent, n_bins=10):
    # YOUR CODE
    return None

def write_audit_row(snap, psi, drift_detected, action_taken):
    # YOUR CODE
    pass

baseline_df = None  # YOUR CODE
recent_df = None    # YOUR CODE
psi_value = None    # YOUR CODE

drift = None        # YOUR CODE
write_audit_row(snap, psi_value, drift, "none")

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
pd.read_parquet(
    f"s3://{SHARED_BUCKET}/{AUDIT_PREFIX}/run_date={today}/"
).sort_values("run_timestamp", ascending=False).head(3)

In [ ]:
# SAFETY-NET for Lab 3.
if 'psi_value' not in dir() or psi_value is None:
    import math
    print("Using Lab 3 safety-net.")

    def psi_amount(baseline, recent, n_bins=10):
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(baseline["amount"].to_numpy(), qs)
        edges[0] = -np.inf
        edges[-1] = np.inf

        b_counts, _ = np.histogram(baseline["amount"].to_numpy(), bins=edges)
        r_counts, _ = np.histogram(recent["amount"].to_numpy(), bins=edges)
        b_total = b_counts.sum() or 1
        r_total = r_counts.sum() or 1
        eps = 1e-10
        psi = 0.0
        for i in range(n_bins):
            bp = b_counts[i] / b_total + eps
            rp = r_counts[i] / r_total + eps
            psi += (rp - bp) * math.log(rp / bp)
        return float(psi)

    def write_audit_row(snap, psi, drift_detected, action_taken):
        record = {
            "run_id": str(uuid.uuid4()),
            "run_timestamp": pd.Timestamp(datetime.now(timezone.utc)),
            "rows_processed": int(snap["rows_arrived"]),
            "fraud_rate": float(snap["fraud_rate_recent"]),
            "psi_score": float(psi),
            "drift_detected": bool(drift_detected),
            "action_taken": str(action_taken),
        }
        return write_audit_record(record)

    baseline_df = df
    recent_df = df.loc[recent_mask]
    psi_value = psi_amount(baseline_df, recent_df)
    drift = (not all_passed) or (psi_value > 0.2)
    write_audit_row(snap, psi_value, drift, "none")
    print("psi_value =", round(psi_value, 4), "drift =", drift)

## Think About It

You now write to BOTH MLflow and an S3 Parquet audit table. That feels
redundant. When is each one the right home?

(Hint: MLflow is shaped around RUNS with parameters/metrics/artifacts
and shines for experiment comparison and UI exploration. An S3 Parquet
dataset is shaped around ROWS you can JOIN against transactions,
customer data, or alert history via Athena or pandas. Operationally
you usually need both.)

## Topic 4 - Triggering retraining via the SageMaker Pipelines API

Detecting drift without acting on it is theater. The data engineer's
job is to hand off cleanly: when our signals fire, we start the ML
engineer's retraining pipeline programmatically.

### The call

We use the SageMaker SDK's `Pipeline.start(parameters=...)`. Under the
hood this calls the `StartPipelineExecution` boto3 API on the
`sagemaker` service. Parameters must match the pipeline's declared
`ParameterString` inputs.

```python
from sagemaker.workflow.pipeline import Pipeline

pipe = Pipeline(name="week20-retrain-stub", sagemaker_session=sess)
execution = pipe.start(parameters={
    "TriggerReason": "drift_detected",
    "PsiScore": "0.31",
    "AuditRunId": "<uuid>",
})
print(execution.arn)
```

Response: a `PipelineExecution` whose `arn` looks like
`arn:aws:sagemaker:us-east-1:535146832369:pipeline/week20-retrain-stub/execution/abcdef`.
We record that ARN in our audit table so the ML engineer can follow
the chain back from a retraining run to the data signals that caused
it.

For class, the instructor has pre-created a tiny stub pipeline
`week20-retrain-stub` that just echoes its parameters. Your
SageMaker execution role already has
`sagemaker:StartPipelineExecution` on it.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

retrain_pipeline = Pipeline(
    name=RETRAIN_PIPELINE_NAME,
    sagemaker_session=sess,
)

def trigger_retraining(reason: str, psi: float, audit_run_id: str):
    execution = retrain_pipeline.start(parameters={
        "TriggerReason": reason,
        "PsiScore": f"{psi:.4f}",
        "AuditRunId": audit_run_id,
    })
    return execution.arn

demo_arn = trigger_retraining("demo", 0.0, "demo")
print("Started:", demo_arn)

## Lab 4 - Glue everything (15 minutes)

Implement `pipeline_run()` that ties the four topics together:

1. Compute the freshness snapshot (reuse `freshness_snapshot`).
2. Run the quality gate (reuse `run_quality_gate`).
3. Compute PSI on `amount` (reuse `psi_amount`).
4. Decide an `action_taken`:
   - `none` if all checks passed and PSI <= 0.2.
   - `alerted` if any check failed OR PSI > 0.2 but PSI <= 0.4.
   - `retrain_triggered` if PSI > 0.4 OR more than one gate check failed.
5. If `action_taken == "retrain_triggered"`, call `trigger_retraining`
   with `reason="drift_detected"` and the PSI score. Use the returned
   execution ARN to overwrite `action_taken` as
   `f"retrain_triggered:{arn}"` before you write the audit row.
6. Write the audit row.

Return the final dict.

You can call your own `pipeline_run()` at the end of the cell to see
it behave end-to-end. Because the source CSV is mostly stable, you
will typically see `action_taken="none"`. That is correct; the
alerting and retraining branches are exercised by the homework data.

In [ ]:
def pipeline_run() -> dict:
    # YOUR CODE
    return None

final = pipeline_run()
print(final)

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
pd.read_parquet(
    f"s3://{SHARED_BUCKET}/{AUDIT_PREFIX}/run_date={today}/"
).sort_values("run_timestamp", ascending=False).head(5)

## Recap

You shipped four pipeline-side MLOps building blocks today, all in
SageMaker Studio with no Spark and no Delta:

- **Freshness monitor** on the fraud CSV in S3, logged to MLflow.
- **Quality gate** with four pandas assertions, halts on bad data.
- **Audit log** as partitioned Parquet under
  `s3://bread-academy-week19-shared/week20/pipeline_audit_log/`.
- **Auto retraining trigger** via the SageMaker Pipelines SDK.

Notice what you did NOT do: you did not retrain the model, you did
not touch the `week19-fraud-endpoint`, you did not edit the model
code. As the data engineer you OWNED the pipeline observability and
HANDED OFF to the model owner. That separation of duties is what
makes MLOps survive contact with production.

## Homework (async, ~45 min)

1. Add a fifth quality check: schema check. Read the column-and-dtype
   tuple of the CSV once and store it as a JSON file at
   `s3://bread-academy-week19-shared/week20/expected_schema.json`. On
   every run, assert that the current schema is identical. Add it to
   `run_quality_gate`.
2. Compute PSI on a categorical column (`merchant_category`). PSI on
   categorical is the same formula but bins are category values, not
   quantiles. Add this as a second drift signal alongside `psi_amount`.
3. Build a small Athena query (or pandas + s3fs scan) against the
   audit dataset that returns the last 7 days of runs, with one row
   per day showing `n_runs`, `n_drift_detected`,
   `n_retrain_triggered`, `avg_psi`. This is the kind of pane an
   on-call data engineer wants pinned on a dashboard.

## Further reading

- SageMaker Pipelines start execution:
  docs.aws.amazon.com/sagemaker/latest/dg/pipelines-run.html
- pandas.read_parquet with S3:
  pandas.pydata.org/docs/reference/api/pandas.read_parquet.html
- Population Stability Index reference: see the Week 19 reading list.
- S3 versioning for audit immutability:
  docs.aws.amazon.com/AmazonS3/latest/userguide/Versioning.html